## Download IMDB directly and save it as a CSV

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"

import time
import pandas as pd
from huggingface_hub import hf_hub_download

os.makedirs("data", exist_ok=True)
t0 = time.time()
path = hf_hub_download("stanfordnlp/imdb", "plain_text/train-00000-of-00001.parquet",
                        repo_type="dataset")
print(f"downloaded in {time.time() - t0:.1f}s")

df = pd.read_parquet(path)
df[["text"]].to_csv("data/corpus.csv", index=False)
print(len(df), "reviews saved to data/corpus.csv")

C:\Users\HAFSSA\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


downloaded in 4.6s
25000 reviews saved to data/corpus.csv


## Point the config at the local file

In [2]:
%%writefile config.py
DATASET = {"source": "csv", "path": "data/corpus.csv", "text_col": "text"}
MAX_DOCS = 4000
MIN_WORD_COUNT = 5
MAX_VOCAB = 15000
MIN_TRI_COUNT = 2
SEED = 42

Overwriting config.py


## Environment and folders

In [3]:
!pip install -q datasets nltk numpy pandas fastapi "uvicorn[standard]" joblib pydantic
import os
for d in ["src", "app", "web", "models", "results", "data"]:
    os.makedirs(d, exist_ok=True)
open("src/__init__.py", "w").close()
open("app/__init__.py", "w").close()


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## data module and splits

In [4]:
%%writefile src/data.py
import random
import re
import time
from collections import Counter

import config

TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")
SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
HTML_RE = re.compile(r"<[^<>]{0,40}>")          # bounded, so it can never backtrack badly


def normalize(text):
    text = HTML_RE.sub(" ", text)               # html such as <br />
    return text.lower().replace("\u2019", "'")


def tokenize(sentence):
    return TOKEN_RE.findall(normalize(sentence))


def split_sentences(text):
    return [s for s in SPLIT_RE.split(normalize(text)) if s and s.strip()]


def load_texts():
    cfg = config.DATASET
    t0 = time.time()
    if cfg["source"] == "csv":
        import pandas as pd
        texts = pd.read_csv(cfg["path"])[cfg["text_col"]].dropna().astype(str).tolist()
        random.Random(config.SEED).shuffle(texts)
        texts = texts[: config.MAX_DOCS]
    else:
        from datasets import load_dataset
        print("  loading dataset (first run downloads about 80 MB)...", flush=True)
        ds = load_dataset(cfg["name"], split=cfg["split"])
        print(f"  dataset ready: {len(ds):,} rows in {time.time() - t0:.1f}s", flush=True)
        ds = ds.shuffle(seed=config.SEED).select(range(min(config.MAX_DOCS, len(ds))))
        texts = ds[cfg["text_col"]]
        texts = [str(t) for t in texts]
    print(f"  {len(texts):,} documents in memory ({time.time() - t0:.1f}s)", flush=True)
    return texts


def build_splits(train_frac=0.8, val_frac=0.1):
    """Split BY DOCUMENT (80/10/10) so sentences of one document never straddle two splits."""
    t0 = time.time()
    docs = []
    for text in load_texts():
        sents = [tokenize(s) for s in split_sentences(text)]
        sents = [s for s in sents if 3 <= len(s) <= 60]
        if sents:
            docs.append(sents)
    print(f"  tokenized {len(docs):,} documents ({time.time() - t0:.1f}s total)", flush=True)
    a, b = int(len(docs) * train_frac), int(len(docs) * (train_frac + val_frac))
    flat = lambda ds: [s for d in ds for s in d]
    return flat(docs[:a]), flat(docs[a:b]), flat(docs[b:])


def build_vocab(train):
    counts = Counter(w for s in train for w in s)
    words = [w for w, c in counts.most_common(config.MAX_VOCAB) if c >= config.MIN_WORD_COUNT]
    return words, counts

Overwriting src/data.py


In [7]:
import os, time
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
os.environ["HF_HUB_DISABLE_XET"] = "1"        # avoids stalls on some networks
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"

import urllib.request
t0 = time.time()
try:
    r = urllib.request.urlopen("https://huggingface.co", timeout=15)
    print("huggingface.co reachable, status", r.status, f"in {time.time()-t0:.1f}s")
except Exception as e:
    print("cannot reach huggingface.co:", repr(e))

huggingface.co reachable, status 200 in 0.3s


In [5]:
import sys, importlib
sys.path.insert(0, ".")
import config
from src import data
importlib.reload(data)

train, val, test = data.build_splits()
words, counts = data.build_vocab(train)
n_tok = lambda ss: sum(len(s) for s in ss)
print(f"train {len(train):,} sent / {n_tok(train):,} tok | val {len(val):,} | test {len(test):,} sent / {n_tok(test):,} tok")
print(f"vocabulary: {len(words):,} words")
print("example:", " ".join(train[0]))

  4,000 documents in memory (0.5s)
  tokenized 3,967 documents (1.5s total)
train 37,212 sent / 688,548 tok | val 4,677 | test 4,665 sent / 85,639 tok
vocabulary: 8,872 words
example: eric clapton jack bruce and ginger baker re unite to play all their songs from years ago when they formed a trio called cream those were the psychedelic days of england and america and these guys looked it all skinny very long hair wild clothes and loud music


## n-gram module (Kneser-Ney and add-k baseline) and training

In [6]:
%%writefile src/ngram.py
import math
from bisect import bisect_left

import numpy as np

UNK, EOS, BOS = 0, 1, 2


class Vocab:
    def __init__(self, words):
        self.itos = ["<unk>", "</s>", "<s>"] + list(words)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.V = len(self.itos)
        real = sorted((w, i) for i, w in enumerate(self.itos) if i > 2)
        self._sorted_words = [w for w, _ in real]
        self._sorted_ids = np.array([i for _, i in real], dtype=np.int64)

    def encode(self, tokens):
        g = self.stoi.get
        return [g(t, UNK) for t in tokens]

    def with_prefix(self, prefix):
        lo = bisect_left(self._sorted_words, prefix)
        hi = bisect_left(self._sorted_words, prefix + "\uffff")
        return self._sorted_ids[lo:hi]


def _discount(counts):
    n1, n2 = int((counts == 1).sum()), int((counts == 2).sum())
    return min(0.95, n1 / (n1 + 2 * n2)) if n1 else 0.75


class KneserNey:
    """Interpolated Kneser-Ney trigram. dist(ctx) returns the FULL next-word distribution (numpy vector).
    Highest order uses raw counts, lower orders use continuation counts."""

    def __init__(self, sentences, vocab, min_tri=1):
        V = self.V = vocab.V
        self.vocab = vocab
        a, b, c = [], [], []
        for s in sentences:
            ids = [BOS, BOS] + vocab.encode(s) + [EOS]
            a += ids[:-2]; b += ids[1:-1]; c += ids[2:]
        a, b, c = (np.asarray(x, dtype=np.int64) for x in (a, b, c))
        keys, cnt = np.unique((a * V + b) * V + c, return_counts=True)
        ua, ub, uc = keys // (V * V), (keys // V) % V, keys % V
        self.D3 = _discount(cnt)

        # level 2: N1+(. v w) from ALL trigram types
        bk, bcnt = np.unique(ub * V + uc, return_counts=True)
        bv, bw = bk // V, bk % V
        self.D2 = _discount(bcnt)
        self.bi_w, self.bi_c = bw, bcnt.astype(float)
        self.bi_index = self._index(bv, self.bi_c)

        # level 1: N1+(. w), interpolated with a uniform distribution
        cont1 = np.bincount(bw, minlength=V).astype(float)
        self.D1 = _discount(cont1[cont1 > 0])
        T, U = cont1.sum(), (cont1 > 0).sum()
        valid = np.ones(V, bool); valid[BOS] = False
        p1 = np.maximum(cont1 - self.D1, 0) / T
        p1[valid] += self.D1 * U / T / valid.sum()
        self.p1 = p1

        # level 3: raw counts, pruned
        keep = cnt >= min_tri
        ua, ub, uc, cnt = ua[keep], ub[keep], uc[keep], cnt[keep].astype(float)
        self.tri_w, self.tri_c = uc, cnt
        self.tri_index = self._index(ua * V + ub, cnt)

    @staticmethod
    def _index(ctx_sorted, cnt):
        uniq, starts = np.unique(ctx_sorted, return_index=True)
        ends = np.append(starts[1:], len(ctx_sorted))
        totals = np.add.reduceat(cnt, starts)
        return {int(k): (int(s), int(e), float(t), int(e - s))
                for k, s, e, t in zip(uniq, starts, ends, totals)}

    def dist(self, ctx):
        """ctx: list of word ids (only the last two are used). Returns (probabilities, order used)."""
        v = ctx[-1] if len(ctx) >= 1 else BOS
        u = ctx[-2] if len(ctx) >= 2 else BOS
        p, level = self.p1, 1
        e = self.bi_index.get(v)
        if e:
            s, t, tot, types = e
            q = p * (self.D2 * types / tot)
            q[self.bi_w[s:t]] += (self.bi_c[s:t] - self.D2) / tot
            p, level = q, 2
        e = self.tri_index.get(u * self.V + v)
        if e:
            s, t, tot, types = e
            q = p * (self.D3 * types / tot)
            q[self.tri_w[s:t]] += (self.tri_c[s:t] - self.D3) / tot
            p, level = q, 3
        return p, level


class AddK:
    """Plain add-k n-gram (baseline). prob(ctx_ids, w) = (c(ctx w)+k) / (c(ctx)+k V)."""

    def __init__(self, sentences, vocab, n=3, k=0.1):
        V = self.V = vocab.V
        self.n, self.k = n, k
        cols = [[] for _ in range(n)]
        for s in sentences:
            ids = [BOS] * (n - 1) + vocab.encode(s) + [EOS]
            L = len(ids)
            for j in range(n):
                cols[j] += ids[j:L - (n - 1 - j)]
        full = np.zeros(len(cols[0]), dtype=np.int64)
        for col in cols:
            full = full * V + np.asarray(col, dtype=np.int64)
        kf, cf = np.unique(full, return_counts=True)
        kc, cc = np.unique(full // V, return_counts=True)
        self.full = dict(zip(kf.tolist(), cf.tolist()))
        self.ctx = dict(zip(kc.tolist(), cc.tolist()))

    def prob(self, ctx, w):
        ck = 0
        for x in ([BOS] * (self.n - 1) + list(ctx))[len(ctx):]:
            ck = ck * self.V + x
        num = self.full.get(ck * self.V + w, 0) + self.k
        den = self.ctx.get(ck, 0) + self.k * self.V
        return num / den

Writing src/ngram.py


In [7]:
import importlib
from src import ngram
importlib.reload(ngram)
from src.ngram import Vocab, KneserNey

vocab = Vocab(words)
kn = KneserNey(train, vocab, min_tri=config.MIN_TRI_COUNT)
print(f"V={vocab.V:,} | discounts D1={kn.D1:.3f} D2={kn.D2:.3f} D3={kn.D3:.3f} | "
      f"trigram contexts kept: {len(kn.tri_index):,} | bigram contexts: {len(kn.bi_index):,}")
p, level = kn.dist(vocab.encode(["this", "movie"]))
print("distribution sums to", round(float(p.sum()), 6), "| order used:", level)

V=8,875 | discounts D1=0.211 D2=0.739 D3=0.858 | trigram contexts kept: 24,698 | bigram contexts: 8,874
distribution sums to 1.0 | order used: 3


In [9]:
import os
print(os.getcwd())
print(sorted(os.listdir("src")))

c:\Users\HAFSSA\Downloads\tp 2 smart keyboard\tp 2 smart keyboard\02_smart_keyboard_v1
['__init__.py', '__pycache__', 'data.py', 'evaluate.py', 'ngram.py', 'pipeline.py']


In [10]:
%%writefile src/hmm.py
from collections import Counter

import numpy as np

from src.ngram import UNK, EOS, BOS


class POSHMM:
    """First-order HMM over part-of-speech tags.
    - viterbi(words): most likely tag sequence
    - next_dist(words): P(next tag or END | words so far) with the forward algorithm
    Emission likelihoods are used through r(t|w) = P(t|w) / P(t), proportional to P(w|t) for a fixed word,
    so unknown words can be handled with a suffix model."""

    def __init__(self, tagged_sents, alpha=0.1):
        self.alpha = alpha
        self.tags = sorted({t for s in tagged_sents for _, t in s})
        T = self.T = len(self.tags)
        tid = {t: i for i, t in enumerate(self.tags)}
        trans = np.zeros((T + 1, T + 1))          # rows: tags + START, cols: tags + END
        tagc = np.zeros(T)
        lex = {}
        for s in tagged_sents:
            prev = T
            for w, t in s:
                j = tid[t]
                trans[prev, j] += 1
                prev = j
                tagc[j] += 1
                lex.setdefault(w, np.zeros(T))[j] += 1
            trans[prev, T] += 1
        n_tok = tagc.sum()
        self.A = (trans + 0.01) / (trans + 0.01).sum(1, keepdims=True)
        self.logA = np.log(self.A)
        self.tok_prior = tagc / n_tok
        self.end_prior = len(tagged_sents) / n_tok
        self.lex = lex
        self.suf = {1: {}, 2: {}, 3: {}}           # suffix statistics from rare words
        for w, c in lex.items():
            if c.sum() <= 5:
                for n in (1, 2, 3):
                    if len(w) > n:
                        self.suf[n].setdefault(w[-n:], np.zeros(T))
                        self.suf[n][w[-n:]] += c
        self._cache = {}

    # ---- emissions
    def posterior(self, w):
        c = self.lex.get(w)
        if c is None:
            for n in (3, 2, 1):
                s = self.suf[n].get(w[-n:]) if len(w) > n else None
                if s is not None and s.sum() >= 5:
                    c = s
                    break
        if c is None:
            return self.tok_prior
        return (c + self.alpha * self.tok_prior) / (c.sum() + self.alpha)

    def ratio(self, w):
        r = self._cache.get(w)
        if r is None:
            r = self.posterior(w) / self.tok_prior
            self._cache[w] = r
        return r

    # ---- inference
    def next_dist(self, words):
        T = self.T
        if not words:
            return self.A[T].copy()
        a = self.A[T, :T] * self.ratio(words[0])
        a /= a.sum()
        for w in words[1:]:
            a = (a @ self.A[:T, :T]) * self.ratio(w)
            a /= a.sum()
        return a @ self.A[:T]                      # length T+1 (last entry = END)

    def viterbi(self, words):
        if not words:
            return []
        T = self.T
        delta = self.logA[T, :T] + np.log(self.ratio(words[0]))
        back = []
        for w in words[1:]:
            scores = delta[:, None] + self.logA[:T, :T]
            back.append(scores.argmax(0))
            delta = scores.max(0) + np.log(self.ratio(w))
        best = int(delta.argmax())
        path = [best]
        for b in reversed(back):
            best = int(b[best])
            path.append(best)
        return [self.tags[i] for i in reversed(path)]

    def vocab_matrix(self, vocab):
        """F[w, t'] = P(t'|w)/P(t')  (END column for </s>). f = F @ next_dist is the syntactic
        compatibility of every vocabulary word with the next position."""
        F = np.ones((vocab.V, self.T + 1))
        for i, w in enumerate(vocab.itos):
            if i in (UNK, BOS):
                continue
            if i == EOS:
                F[i] = 0.0
                F[i, self.T] = 1.0 / self.end_prior
            else:
                F[i, :self.T] = self.ratio(w)
                F[i, self.T] = 0.0
        return F

    def evaluate(self, tagged_sents):
        c = Counter()
        default = self.tags[int(np.argmax(self.tok_prior))]
        for s in tagged_sents:
            pred = self.viterbi([w for w, _ in s])
            for (w, gold), p in zip(s, pred):
                known = w in self.lex
                mft = self.tags[int(np.argmax(self.lex[w]))] if known else default
                c["n"] += 1
                c["ok"] += p == gold

Writing src/hmm.py


In [11]:
print(sorted(os.listdir("src")))

['__init__.py', '__pycache__', 'data.py', 'evaluate.py', 'hmm.py', 'ngram.py', 'pipeline.py']


In [13]:
print(open("src/hmm.py", encoding="utf-8").read()[-700:])

if i in (UNK, BOS):
                continue
            if i == EOS:
                F[i] = 0.0
                F[i, self.T] = 1.0 / self.end_prior
            else:
                F[i, :self.T] = self.ratio(w)
                F[i, self.T] = 0.0
        return F

    def evaluate(self, tagged_sents):
        c = Counter()
        default = self.tags[int(np.argmax(self.tok_prior))]
        for s in tagged_sents:
            pred = self.viterbi([w for w, _ in s])
            for (w, gold), p in zip(s, pred):
                known = w in self.lex
                mft = self.tags[int(np.argmax(self.lex[w]))] if known else default
                c["n"] += 1
                c["ok"] += p == gold



In [14]:
src = open("src/hmm.py", encoding="utf-8").read()
head = src[: src.index("    def evaluate")]

evaluate_method = '''    def evaluate(self, tagged_sents):
        c = Counter()
        default = self.tags[int(np.argmax(self.tok_prior))]
        for s in tagged_sents:
            pred = self.viterbi([w for w, _ in s])
            for (w, gold), p in zip(s, pred):
                known = w in self.lex
                mft = self.tags[int(np.argmax(self.lex[w]))] if known else default
                c["n"] += 1
                c["ok"] += p == gold
                c["base"] += mft == gold
                c["kn"] += known
                c["kok"] += known and p == gold
                c["un"] += not known
                c["uok"] += (not known) and p == gold
        return {"accuracy": c["ok"] / c["n"], "baseline": c["base"] / c["n"],
                "accuracy_known": c["kok"] / max(1, c["kn"]),
                "accuracy_unknown": c["uok"] / max(1, c["un"]),
                "unknown_share": c["un"] / c["n"], "n_tokens": c["n"]}
'''
open("src/hmm.py", "w", encoding="utf-8").write(head + evaluate_method)
print("patched")
print(open("src/hmm.py", encoding="utf-8").read()[-300:])

patched
not known) and p == gold
        return {"accuracy": c["ok"] / c["n"], "baseline": c["base"] / c["n"],
                "accuracy_known": c["kok"] / max(1, c["kn"]),
                "accuracy_unknown": c["uok"] / max(1, c["un"]),
                "unknown_share": c["un"] / c["n"], "n_tokens": c["n"]}



In [15]:
import importlib
from src import hmm as hmm_mod
importlib.reload(hmm_mod)
from src.hmm import POSHMM

hmm = POSHMM(tagged[:cut])
hmm_eval = hmm.evaluate(tagged[cut:])
print("tags:", hmm.tags)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in hmm_eval.items()})
print(list(zip("the acting in this film was brilliant".split(), hmm.viterbi("the acting in this film was brilliant".split()))))

tags: ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
{'accuracy': 0.9636, 'baseline': 0.9391, 'accuracy_known': 0.966, 'accuracy_unknown': 0.837, 'unknown_share': 0.0189, 'n_tokens': 99051}
[('the', 'DET'), ('acting', 'VERB'), ('in', 'ADP'), ('this', 'DET'), ('film', 'NOUN'), ('was', 'VERB'), ('brilliant', 'ADJ')]


In [16]:
import random, re
import nltk
nltk.download("brown", quiet=True)
nltk.download("universal_tagset", quiet=True)
from nltk.corpus import brown
from src import hmm as hmm_mod
importlib.reload(hmm_mod)
from src.hmm import POSHMM

ok_word = re.compile(r"^[a-z]+(?:'[a-z]+)?$")
tagged = []
for sent in brown.tagged_sents(tagset="universal"):
    s = [(w.lower(), t) for w, t in sent if t != "." and ok_word.match(w.lower())]
    if len(s) >= 3:
        tagged.append(s)
random.Random(config.SEED).shuffle(tagged)
cut = int(len(tagged) * 0.9)

hmm = POSHMM(tagged[:cut])
hmm_eval = hmm.evaluate(tagged[cut:])
print("tags:", hmm.tags)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in hmm_eval.items()})
print(list(zip("the acting in this film was brilliant".split(), hmm.viterbi("the acting in this film was brilliant".split()))))

tags: ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
{'accuracy': 0.9636, 'baseline': 0.9391, 'accuracy_known': 0.966, 'accuracy_unknown': 0.837, 'unknown_share': 0.0189, 'n_tokens': 99051}
[('the', 'DET'), ('acting', 'VERB'), ('in', 'ADP'), ('this', 'DET'), ('film', 'NOUN'), ('was', 'VERB'), ('brilliant', 'ADJ')]


## evaluation (tuning on validation, one report on test)

In [18]:
%%writefile src/evaluate.py
import json
import math
import random
import time

from src.ngram import UNK, BOS, AddK
from src.engine import Predictor


def positions(sentences, limit, seed=0):
    pos = [(s, i) for s in sentences for i in range(len(s) + 1)]
    random.Random(seed).shuffle(pos)
    return pos[:limit]


def target(s, i):
    return s[i] if i < len(s) else "</s>"


def scan(pred, pos, betas):
    """One pass over positions; log-likelihood and rank metrics for every beta (beta = HMM weight)."""
    res = {b: dict(ll=0.0, n=0, h1=0, h3=0, h5=0, rr=0.0, inv=0) for b in betas}
    for s, i in pos:
        ctx = s[:i]
        wid = pred.vocab.stoi.get(target(s, i), UNK)
        base, _, _, f = pred._parts(ctx)
        for b in betas:
            p = pred._combine(base, f, True, b)
            r = res[b]
            r["ll"] += math.log(p[wid] / p.sum())
            r["n"] += 1
            if wid != UNK:
                q = p.copy()
                q[UNK] = 0.0
                q[BOS] = 0.0
                rank = int((q > q[wid]).sum()) + 1
                r["inv"] += 1
                r["h1"] += rank <= 1
                r["h3"] += rank <= 3
                r["h5"] += rank <= 5
                r["rr"] += 1.0 / rank
    return res


def summarize(r):
    return {"perplexity": math.exp(-r["ll"] / r["n"]), "top1": r["h1"] / r["inv"],
            "top3": r["h3"] / r["inv"], "top5": r["h5"] / r["inv"], "mrr": r["rr"] / r["inv"]}


def keystroke_savings(pred, pos, use_hmm, k=3, n=800):
    """Share of keystrokes saved if the user taps a suggestion as soon as it appears (space counts as 1)."""
    saved = total = 0
    for s, i in pos[:n]:
        if i >= len(s) or s[i] not in pred.vocab.stoi:
            continue
        w, full = s[i], len(s[i]) + 1
        typed = full
        for p in range(len(w)):
            if w in pred.topk_words(s[:i], w[:p], k, use_hmm):
                typed = p + 1
                break
        saved += full - min(typed, full)
        total += full
    return saved / total


def addk_ppl(model, pos, vocab):
    ll = 0.0
    for s, i in pos:
        ctx = vocab.encode(s[max(0, i - 2):i])
        ll += math.log(model.prob(ctx, vocab.stoi.get(target(s, i), UNK)))
    return math.exp(-ll / len(pos))


def run(train, val, test, vocab, kn, hmm, hmm_eval, val_n=6000, test_n=15000, out="results/metrics.json"):
    pred = Predictor(kn, hmm, vocab, beta=0.0)
    vp, tp = positions(val, val_n, 1), positions(test, test_n, 2)

    # 1) tune the HMM weight on VALIDATION only
    betas = [0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
    v = scan(pred, vp, betas)
    sweep = [{"beta": b, **summarize(v[b])} for b in betas]
    best = min(sweep, key=lambda r: r["perplexity"])["beta"]
    pred.beta = best

    # 2) add-k baselines, k tuned on validation
    models = []
    for n in (1, 2, 3):
        m = AddK(train, vocab, n, 1.0)
        scores = {}
        for k in (0.001, 0.01, 0.05, 0.1, 0.5, 1.0):
            m.k = k
            scores[k] = addk_ppl(m, vp, vocab)
        m.k = min(scores, key=scores.get)
        models.append({"name": f"{['Unigram', 'Bigram', 'Trigram'][n - 1]}, add-k (k={m.k})",
                       "perplexity": addk_ppl(m, tp, vocab), "top1": None, "top3": None,
                       "top5": None, "mrr": None, "kss": None})

    # 3) final test report (once)
    t = scan(pred, tp, [0, best])
    kss0 = keystroke_savings(pred, tp, use_hmm=False)
    kss1 = keystroke_savings(pred, tp, use_hmm=True)
    models.append({"name": "Kneser-Ney trigram", **summarize(t[0]), "kss": kss0})
    models.append({"name": f"Kneser-Ney trigram + HMM (beta={best})", **summarize(t[best]), "kss": kss1})

    t0 = time.perf_counter()
    for s, i in tp[:300]:
        pred.suggest(" ".join(s[:i]) + " ", 3)
    latency = (time.perf_counter() - t0) / 300 * 1000

    n_test = sum(len(s) for s in test)
    metrics = {
        "dataset": {"name": __import__("config").DATASET.get("name", "csv"),
                    "train_tokens": sum(len(s) for s in train), "test_tokens": n_test,
                    "vocab": vocab.V,
                    "oov_rate": sum(w not in vocab.stoi for s in test for w in s) / n_test,
                    "eval_positions": len(tp)},
        "models": models, "beta_sweep": sweep, "best_beta": best, "hmm": hmm_eval,
        "latency_ms": latency,
    }
    with open(out, "w") as fh:
        json.dump(metrics, fh, indent=2)
    return metrics

Overwriting src/evaluate.py


In [20]:
%%writefile src/engine.py
import math
import re
import time
from collections import Counter

import numpy as np

from src.data import tokenize, split_sentences
from src.ngram import UNK, EOS, BOS

WORD = re.compile(r"[a-z]+(?:'[a-z]+)?")
LEVELS = {1: "unigram", 2: "bigram", 3: "trigram"}


def parse(text):
    """-> (completed words of the current sentence, prefix of the word being typed)."""
    cur = re.split(r"[.!?\n]+", text.lower().replace("\u2019", "'"))[-1]
    toks = WORD.findall(cur)
    if toks and (cur[-1].isalpha() or cur[-1] == "'"):
        return toks[:-1], toks[-1]
    return toks, ""


class UserCache:
    """Per-user adaptive model over WORD STRINGS, so words outside the vocabulary (names, slang) are learned too."""
    MAX_SHARE, PRIOR = 0.5, 3.0

    def __init__(self):
        self.tri, self.bi, self.uni = {}, {}, Counter()
        self.n_tokens = 0

    def learn(self, text, vocab):
        new, n = set(), 0
        for sent in split_sentences(text):
            hist = ["<s>", "<s>"]
            for w in tokenize(sent):
                self.tri.setdefault((hist[-2], hist[-1]), Counter())[w] += 1
                self.bi.setdefault(hist[-1], Counter())[w] += 1
                self.uni[w] += 1
                if w not in vocab.stoi:
                    new.add(w)
                hist.append(w)
                n += 1
        self.n_tokens += n
        return {"tokens": n, "n_new_words": len(new), "new_words": sorted(new)[:12],
                "profile_tokens": self.n_tokens}

    def blend(self, p, ctx, prefix, vocab):
        u, v = (["<s>", "<s>"] + ctx)[-2:]
        sources = [self.tri.get((u, v)), self.bi.get(v)]
        if prefix:
            sources.append(self.uni)
        cnt, tot, src = None, 0, None
        for cand in sources:
            if not cand:
                continue
            sel = {w: c for w, c in cand.items() if w.startswith(prefix)}
            if sel:
                cnt, tot, src = sel, sum(sel.values()), cand
                break
        if not cnt:
            return p, [], set()
        m = min(self.MAX_SHARE, tot / (tot + self.PRIOR)) * (0.5 if src is self.uni else 1.0)
        q, extra = p * (1 - m), []
        for w, c in cnt.items():
            i = vocab.stoi.get(w)
            if i is None:
                extra.append((w, m * c / tot))
            else:
                q[i] += m * c / tot
        z = q.sum() + sum(x for _, x in extra)
        if z > 0:
            q = q / z
            extra = [(w, x / z) for w, x in extra]
        return q, extra, set(cnt)


class Predictor:
    def __init__(self, kn, hmm, vocab, beta=0.4):
        self.kn, self.hmm, self.vocab, self.beta = kn, hmm, vocab, beta
        self.F = hmm.vocab_matrix(vocab)

    # ---- distribution pieces
    def _parts(self, ctx):
        base, level = self.kn.dist(self.vocab.encode(ctx))
        nd = self.hmm.next_dist(ctx[-30:])
        f = self.F @ nd
        return base, level, nd, f

    def _combine(self, base, f, use_hmm=True, beta=None):
        beta = self.beta if beta is None else beta
        return base * f ** beta if (use_hmm and beta > 0) else base

    def _mask(self, p, prefix):
        q = p.copy()
        q[UNK] = 0.0
        q[BOS] = 0.0
        if prefix:
            keep = np.zeros_like(q)
            keep[self.vocab.with_prefix(prefix)] = 1.0
            q *= keep
        s = q.sum()
        return q / s if s > 0 else q

    def topk_words(self, ctx, prefix="", k=3, use_hmm=True):
        base, _, _, f = self._parts(ctx)
        q = self._mask(self._combine(base, f, use_hmm), prefix)
        if q.sum() <= 0:
            return []
        kk = min(k, len(q) - 1)
        idx = np.argpartition(-q, kk)[:kk]
        idx = idx[np.argsort(-q[idx])]
        return [self.vocab.itos[i] for i in idx if q[i] > 0]

    # ---- public API
    def suggest(self, text, k=3, user=None, use_hmm=True):
        t0 = time.perf_counter()
        ctx, prefix = parse(text)
        ctx = ctx[-40:]
        base, level, nd, f = self._parts(ctx)
        final = self._mask(self._combine(base, f, use_hmm), prefix)
        plain = self._mask(base, prefix)
        extra, boosted = [], set()
        if user is not None:
            final, extra, boosted = user.blend(final, ctx, prefix, self.vocab)
        cands = []
        if final.sum() > 0:
            kk = min(k, len(final) - 1)
            for i in np.argpartition(-final, kk)[:kk]:
                if final[i] > 0:
                    cands.append((self.vocab.itos[i], float(final[i]), int(i)))
        cands += [(w, float(x), None) for w, x in extra]
        cands.sort(key=lambda c: -c[1])
        T = self.hmm.T
        out = []
        for w, pr, i in cands[:k]:
            eos = i == EOS
            pos = "END" if eos else self.hmm.tags[int(np.argmax(self.hmm.ratio(w) * nd[:T]))]
            out.append({"word": w, "prob": round(pr, 4),
                        "base_prob": round(float(plain[i]), 4) if i is not None else None,
                        "pos": pos, "eos": eos, "personal": w in boosted})
        words = ctx[-20:]
        tags = self.hmm.viterbi(words)
        names = self.hmm.tags + ["END"]
        order = np.argsort(-nd)[:4]
        return {
            "mode": "completion" if prefix else "next_word",
            "prefix": prefix,
            "context": ctx[-6:],
            "order": LEVELS[level],
            "oov": [w for w in ctx if w not in self.vocab.stoi],
            "suggestions": out,
            "tags": [{"word": w, "tag": t} for w, t in zip(words, tags)],
            "next_tags": [{"tag": names[i], "p": round(float(nd[i]), 4)} for i in order],
            "latency_ms": round((time.perf_counter() - t0) * 1000, 2),
        }

    def complete(self, text, length=4, beam=3, use_hmm=True):
        """Beam-search phrase completion (the first word respects the prefix being typed)."""
        ctx, prefix = parse(text)
        ctx = ctx[-40:]
        beams = [(0.0, [], False)]
        for step in range(length):
            nxt = []
            for score, ws, done in beams:
                if done:
                    nxt.append((score, ws, True))
                    continue
                base, _, _, f = self._parts(ctx + ws)
                q = self._mask(self._combine(base, f, use_hmm), prefix if step == 0 else "")
                if q.sum() <= 0:
                    nxt.append((score, ws, True))
                    continue
                for i in np.argpartition(-q, beam)[:beam]:
                    if q[i] <= 0:
                        continue
                    ended = int(i) == EOS
                    nxt.append((score + math.log(q[i]),
                                ws if ended else ws + [self.vocab.itos[i]], ended))
            beams = sorted(nxt, key=lambda b: -b[0])[:beam]
        out, seen = [], set()
        for score, ws, done in sorted(beams, key=lambda b: -b[0] / (len(b[1]) + 1)):
            t = " ".join(ws)
            if t and t not in seen:
                seen.add(t)
                out.append({"text": t, "ended": done, "score": round(score / (len(ws) + 1), 3)})
        return out

Writing src/engine.py


In [21]:
print(sorted(os.listdir("src")))
print(repr(open("src/engine.py", encoding="utf-8").read()[-60:]))

['__init__.py', '__pycache__', 'data.py', 'engine.py', 'evaluate.py', 'hmm.py', 'ngram.py', 'pipeline.py']
'core": round(score / (len(ws) + 1), 3)})\n        return out\n'


In [22]:
from src import engine, evaluate
importlib.reload(engine); importlib.reload(evaluate)
metrics = evaluate.run(train, val, test, vocab, kn, hmm, hmm_eval)

print(f"best beta (chosen on validation): {metrics['best_beta']}")
for r in metrics["models"]:
    f = lambda x, p=False: "-" if x is None else (f"{x*100:.1f}%" if p else f"{x:.2f}")
    print(f"{r['name']:<42} ppl {f(r['perplexity']):>7}  top1 {f(r['top1'],1):>6}  top3 {f(r['top3'],1):>6}  "
            f"top5 {f(r['top5'],1):>6}  mrr {f(r['mrr']):>5}  kss {f(r['kss'],1):>6}")
print("HMM tagging accuracy:", round(metrics["hmm"]["accuracy"], 4), "| latency ms/request:", round(metrics["latency_ms"], 2))

best beta (chosen on validation): 0.2
Unigram, add-k (k=0.001)                   ppl  502.41  top1      -  top3      -  top5      -  mrr     -  kss      -
Bigram, add-k (k=0.01)                     ppl  287.16  top1      -  top3      -  top5      -  mrr     -  kss      -
Trigram, add-k (k=0.01)                    ppl 1209.07  top1      -  top3      -  top5      -  mrr     -  kss      -
Kneser-Ney trigram                         ppl  208.49  top1  18.0%  top3  31.0%  top5  37.1%  mrr  0.27  kss  51.2%
Kneser-Ney trigram + HMM (beta=0.2)        ppl  208.08  top1  18.0%  top3  30.9%  top5  37.0%  mrr  0.27  kss  51.1%
HMM tagging accuracy: 0.9636 | latency ms/request: 0.95


In [32]:
import os, joblib
os.makedirs("models", exist_ok=True)
joblib.dump({"kn": kn, "hmm": hmm, "vocab": vocab, "beta": metrics["best_beta"],
             "meta": {"dataset": metrics["dataset"]["name"], "vocab": vocab.V}},
            "models/engine.joblib", compress=3)
print(f"saved: {os.path.getsize('models/engine.joblib')/1e6:.1f} MB")

saved: 2.3 MB


In [33]:
for p in ["models/engine.joblib", "results/metrics.json", "app/main.py", "web/index.html", "web/style.css", "web/app.js"]:
    print(f"{p:<24}", "ok" if os.path.exists(p) else "MISSING")

models/engine.joblib     ok
results/metrics.json     ok
app/main.py              ok
web/index.html           ok
web/style.css            ok
web/app.js               ok


## API (FastAPI)

In [23]:
%%writefile app/main.py
import json
from collections import OrderedDict
from pathlib import Path
from typing import Optional

import joblib
from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field

from src.engine import Predictor, UserCache

ROOT = Path(__file__).resolve().parent.parent
bundle = joblib.load(ROOT / "models" / "engine.joblib")
engine = Predictor(bundle["kn"], bundle["hmm"], bundle["vocab"], bundle["beta"])
USERS: "OrderedDict[str, UserCache]" = OrderedDict()     # in-memory only; nothing is written to disk

app = FastAPI(title="Smart Keyboard")


def get_user(uid: Optional[str], create: bool = False):
    if not uid:
        return None
    c = USERS.get(uid)
    if c is None and create:
        c = USERS[uid] = UserCache()
        if len(USERS) > 500:
            USERS.popitem(last=False)
    if c is not None:
        USERS.move_to_end(uid)
    return c


class SuggestIn(BaseModel):
    text: str = Field("", max_length=2000)
    k: int = Field(3, ge=1, le=8)
    user_id: Optional[str] = Field(None, max_length=64)
    use_hmm: bool = True
    personalize: bool = True


class CompleteIn(BaseModel):
    text: str = Field("", max_length=2000)
    length: int = Field(4, ge=1, le=8)
    beam: int = Field(3, ge=1, le=5)
    use_hmm: bool = True


class LearnIn(BaseModel):
    text: str = Field(..., max_length=20000)
    user_id: str = Field(..., max_length=64)


class ForgetIn(BaseModel):
    user_id: str = Field(..., max_length=64)


@app.get("/api/health")
def health():
    return {"ok": True}


@app.get("/api/info")
def info():
    return {"vocab": engine.vocab.V, "beta": engine.beta, "tags": engine.hmm.tags, **bundle["meta"]}


@app.post("/api/suggest")
def suggest(body: SuggestIn):
    user = get_user(body.user_id) if body.personalize else None
    return engine.suggest(body.text, body.k, user=user, use_hmm=body.use_hmm)


@app.post("/api/complete")
def complete(body: CompleteIn):
    return {"completions": engine.complete(body.text, body.length, body.beam, body.use_hmm)}


@app.post("/api/learn")
def learn(body: LearnIn):
    return get_user(body.user_id, create=True).learn(body.text, engine.vocab)


@app.post("/api/forget")
def forget(body: ForgetIn):
    USERS.pop(body.user_id, None)
    return {"ok": True}


@app.get("/api/metrics")
def metrics():
    path = ROOT / "results" / "metrics.json"
    if not path.exists():
        raise HTTPException(404, "metrics.json not found")
    return json.loads(path.read_text())


app.mount("/", StaticFiles(directory=ROOT / "web", html=True), name="web")

Writing app/main.py


## front end, HTML

In [24]:
%%writefile web/index.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Smart Keyboard</title>
  <link rel="stylesheet" href="style.css">
</head>
<body>
<header class="topbar">
  <div class="brand">
    <span class="brand-mark" aria-hidden="true"></span>
    <div>
      <strong>Smart Keyboard</strong>
      <span class="brand-sub">Next-word prediction with n-gram and HMM models</span>
    </div>
  </div>
  <nav class="tabs" id="tabs" role="tablist">
    <button class="tab is-active" data-view="writer" role="tab">Writer</button>
    <button class="tab" data-view="analysis" role="tab">Analysis</button>
    <button class="tab" data-view="evaluation" role="tab">Evaluation</button>
    <span class="tab-ink" id="tabInk"></span>
  </nav>
  <div class="status" id="status"><span class="status-dot"></span><span id="statusText">Connecting</span></div>
</header>

<main>
  <!-- WRITER -->
  <section class="view is-active" id="view-writer">
    <div class="layout">
      <div class="stack">
        <div class="card">
          <div class="card-head"><h2>Compose</h2><span class="muted" id="wordCount">0 words</span></div>
          <textarea id="editor" rows="7" spellcheck="false" autofocus
                    placeholder="Start typing. Suggestions follow the cursor."></textarea>
          <div class="suggest-bar" id="slots" aria-live="polite"></div>
          <div class="keys">
            <span><kbd>Tab</kbd> accept first</span>
            <span><kbd>Alt</kbd> + <kbd>1</kbd> to <kbd>5</kbd> choose</span>
          </div>
        </div>
        <div class="card">
          <div class="card-head"><h3>Phrase completions</h3><span class="muted">Beam search, 4 words</span></div>
          <ul class="phrases" id="phrases"></ul>
        </div>
      </div>

      <aside class="stack">
        <div class="card">
          <div class="card-head"><h3>Engine state</h3></div>
          <dl class="kv">
            <div><dt>Mode</dt><dd id="stMode">-</dd></div>
            <div><dt>Context order</dt><dd id="stOrder">-</dd></div>
            <div><dt>Latency</dt><dd id="stLatency">-</dd></div>
          </dl>
          <div class="ctx" id="stContext"></div>
        </div>
        <div class="card">
          <div class="card-head"><h3>Next tag, HMM</h3></div>
          <ul class="rank" id="nextTags"></ul>
        </div>
        <div class="card">
          <div class="card-head"><h3>Options</h3></div>
          <label class="switch"><input type="checkbox" id="optHmm" checked><span class="track"></span><span>Syntactic model (HMM)</span></label>
          <label class="switch"><input type="checkbox" id="optPersonal" checked><span class="track"></span><span>Personal profile</span></label>
          <div class="seg" id="seg">
            <span class="muted">Suggestions</span>
            <button data-k="3" class="is-active">3</button>
            <button data-k="5">5</button>
          </div>
        </div>
        <div class="card">
          <div class="card-head"><h3>Personal profile</h3></div>
          <p class="muted">Learn the vocabulary and phrasing of your text. Stored in memory for this session only.</p>
          <div class="row">
            <button class="btn" id="btnLearn">Learn from text</button>
            <button class="btn btn-ghost" id="btnForget">Reset</button>
          </div>
          <p class="note" id="profileNote">Nothing learned yet.</p>
        </div>
      </aside>
    </div>
  </section>

  <!-- ANALYSIS -->
  <section class="view" id="view-analysis">
    <div class="card">
      <div class="card-head"><h2>Sentence analysis</h2><span class="muted">Viterbi tagging and model comparison</span></div>
      <input id="analysisInput" class="line-input" value="i thought the acting was " spellcheck="false">
      <div class="tag-row" id="tagRow"></div>
      <div class="legend" id="legend"></div>
    </div>
    <div class="grid-2">
      <div class="card"><div class="card-head"><h3>N-gram only</h3><span class="muted">Kneser-Ney</span></div><ul class="rank" id="rankBase"></ul></div>
      <div class="card"><div class="card-head"><h3>N-gram with HMM</h3><span class="muted">Syntactic reweighting</span></div><ul class="rank" id="rankHmm"></ul></div>
    </div>
  </section>

  <!-- EVALUATION -->
  <section class="view" id="view-evaluation">
    <div class="metrics" id="metricCards"></div>
    <div class="card">
      <div class="card-head"><h2>Model comparison</h2><span class="muted" id="datasetLine"></span></div>
      <div class="table-wrap"><table id="modelTable"></table></div>
    </div>
    <div class="grid-2">
      <div class="card"><div class="card-head"><h3>HMM weight (validation)</h3></div><div class="table-wrap"><table id="sweepTable"></table></div></div>
      <div class="card"><div class="card-head"><h3>POS tagger</h3></div><dl class="kv" id="posStats"></dl></div>
    </div>
  </section>
</main>

<script src="app.js"></script>
</body>
</html>

Writing web/index.html


In [25]:
%%writefile web/style.css
:root{
  --bg:#F4F6F9; --surface:#FFFFFF; --ink:#0B1B2B; --muted:#5B6B7C; --line:#DDE3EA;
  --accent:#0B5FFF; --accent-soft:#E7F0FF; --ok:#0E7C66; --bad:#B4233A;
  --radius:10px; --ease:cubic-bezier(.2,.7,.2,1);
}
*{box-sizing:border-box}
html,body{margin:0}
body{background:var(--bg);color:var(--ink);font:15px/1.5 "Inter","Segoe UI",system-ui,-apple-system,Roboto,sans-serif;-webkit-font-smoothing:antialiased}
h2,h3{margin:0;font-weight:600;letter-spacing:-.01em}
h2{font-size:16px} h3{font-size:14px}
.muted{color:var(--muted);font-size:13px}
button{font:inherit;color:inherit}

/* top bar */
.topbar{position:sticky;top:0;z-index:10;display:flex;align-items:center;justify-content:space-between;gap:24px;height:64px;padding:0 28px;background:var(--surface);border-bottom:1px solid var(--line)}
.brand{display:flex;align-items:center;gap:12px}
.brand strong{display:block;font-size:15px}
.brand-sub{display:block;font-size:12px;color:var(--muted)}
.brand-mark{position:relative;width:30px;height:30px;border-radius:8px;background:var(--ink)}
.brand-mark::before,.brand-mark::after{content:"";position:absolute;left:7px;height:3px;border-radius:2px;background:#fff}
.brand-mark::before{top:10px;width:16px}
.brand-mark::after{top:17px;width:10px;background:#5B9BFF}
.tabs{position:relative;display:flex;gap:4px;height:100%}
.tab{appearance:none;border:0;background:none;padding:0 18px;font-weight:500;color:var(--muted);cursor:pointer;transition:color .2s}
.tab:hover,.tab.is-active{color:var(--ink)}
.tab-ink{position:absolute;left:0;bottom:0;height:2px;width:0;background:var(--accent);transition:transform .35s var(--ease),width .35s var(--ease)}
.status{display:flex;align-items:center;gap:8px;font-size:13px;color:var(--muted);min-width:130px;justify-content:flex-end}
.status-dot{width:8px;height:8px;border-radius:50%;background:#9AA8B6;transition:background .3s}
.status.is-ok .status-dot{background:var(--ok);animation:pulse 2.4s ease-out infinite}
.status.is-bad .status-dot{background:var(--bad)}

/* layout */
main{max-width:1180px;margin:0 auto;padding:28px}
.view{display:none}
.view.is-active{display:block;animation:rise .45s var(--ease) both}
.layout{display:grid;grid-template-columns:minmax(0,1.7fr) minmax(280px,1fr);gap:20px;align-items:start}
.stack{display:flex;flex-direction:column;gap:20px}
.grid-2{display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-top:20px}
.card{background:var(--surface);border:1px solid var(--line);border-radius:var(--radius);padding:20px;box-shadow:0 1px 2px rgba(11,27,43,.04)}
.card-head{display:flex;align-items:baseline;justify-content:space-between;gap:12px;margin-bottom:14px}
#view-analysis .card + .grid-2{margin-top:20px}

/* editor */
#editor{width:100%;resize:vertical;min-height:150px;padding:16px;border:1px solid var(--line);border-radius:8px;background:#FBFCFD;font:18px/1.65 inherit;font-family:inherit;color:var(--ink);transition:border-color .2s,box-shadow .2s,background .2s}
#editor:focus,.line-input:focus{outline:0;border-color:var(--accent);background:#fff;box-shadow:0 0 0 4px var(--accent-soft)}
.suggest-bar{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin-top:14px}
.slot{appearance:none;display:flex;flex-direction:column;gap:10px;text-align:left;padding:12px 14px;background:var(--surface);border:1px solid var(--line);border-radius:var(--radius);cursor:pointer;transition:border-color .2s,transform .18s var(--ease),box-shadow .2s,opacity .25s}
.slot:first-child{border-color:#B9D0FF;background:var(--accent-soft)}
.slot:hover{border-color:var(--accent);transform:translateY(-2px);box-shadow:0 8px 18px rgba(11,27,43,.09)}
.slot:active{transform:scale(.98)}
.slot.is-empty{opacity:.4;pointer-events:none}
.slot.is-personal{box-shadow:inset 3px 0 0 var(--ok)}
.slot-top{display:flex;align-items:center;justify-content:space-between;gap:8px;min-height:26px}
.slot-word{font-size:17px;font-weight:600;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.slot-meta{display:flex;justify-content:space-between;font-size:12px;color:var(--muted)}
.keys{display:flex;gap:18px;margin-top:12px;font-size:12px;color:var(--muted)}
kbd{padding:1px 6px;border:1px solid var(--line);border-bottom-width:2px;border-radius:5px;background:#fff;font:11px/1.4 inherit;font-family:inherit;color:var(--ink)}

/* bars and ranks */
.bar{display:block;height:4px;border-radius:2px;background:#E4E9EF;overflow:hidden}
.bar i{display:block;height:100%;width:calc(var(--w,0) * 100%);background:var(--accent);border-radius:2px;transition:width .5s var(--ease)}
.rank{list-style:none;margin:0;padding:0;display:flex;flex-direction:column;gap:10px}
.rank li{display:grid;grid-template-columns:minmax(70px,1.1fr) 2fr 52px;align-items:center;gap:12px;font-size:14px;animation:rise .3s var(--ease) both;animation-delay:calc(var(--i,0) * 35ms)}
.rank-label{display:flex;align-items:center;gap:8px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.rank-val{text-align:right;color:var(--muted);font-variant-numeric:tabular-nums;font-size:13px}
.phrases{list-style:none;margin:0;padding:0;display:flex;flex-direction:column;gap:8px}
.phrases button{appearance:none;width:100%;display:flex;justify-content:space-between;gap:12px;padding:10px 14px;border:1px solid var(--line);border-radius:8px;background:#fff;text-align:left;cursor:pointer;transition:border-color .2s,background .2s,transform .15s var(--ease);animation:rise .3s var(--ease) both;animation-delay:calc(var(--i,0) * 40ms)}
.phrases button:hover{border-color:var(--accent);background:var(--accent-soft);transform:translateX(3px)}
.phrases .ended{color:var(--muted);font-size:12px}

/* tags */
.tag{--tc:#475569;--tb:#EDF1F5;display:inline-block;padding:1px 8px;border-radius:999px;background:var(--tb);color:var(--tc);font-size:11px;font-weight:600;letter-spacing:.02em;line-height:1.7}
.tag[hidden]{display:none}
[data-tag="NOUN"]{--tc:#0B4FCC;--tb:#E7F0FF}
[data-tag="VERB"]{--tc:#0E7C66;--tb:#E1F4EE}
[data-tag="ADJ"]{--tc:#9A5A06;--tb:#FCEFD9}
[data-tag="ADV"]{--tc:#A3224A;--tb:#FCE7ED}
[data-tag="DET"]{--tc:#475569;--tb:#EDF1F5}
[data-tag="ADP"]{--tc:#1B6A88;--tb:#E2F1F7}
[data-tag="PRON"]{--tc:#6B5A1E;--tb:#F6F0DA}
[data-tag="NUM"]{--tc:#7A4B1E;--tb:#F6ECE2}
[data-tag="END"]{--tc:#fff;--tb:#0B1B2B}

.kv{margin:0;display:flex;flex-direction:column;gap:8px}
.kv div{display:flex;justify-content:space-between;gap:12px;font-size:14px}
.kv dt{color:var(--muted)} .kv dd{margin:0;font-weight:500;text-align:right}
.ctx{display:flex;flex-wrap:wrap;gap:6px;margin-top:14px}
.chip{padding:2px 9px;border-radius:6px;background:#EDF1F5;font-size:13px;animation:rise .25s var(--ease) both}
.chip.is-oov{background:#FCE7ED;color:var(--bad)}

/* controls */
.switch{display:flex;align-items:center;gap:12px;padding:6px 0;cursor:pointer;font-size:14px}
.switch input{position:absolute;opacity:0}
.track{position:relative;width:36px;height:20px;border-radius:10px;background:#C9D2DC;transition:background .25s;flex:none}
.track::after{content:"";position:absolute;top:2px;left:2px;width:16px;height:16px;border-radius:50%;background:#fff;box-shadow:0 1px 2px rgba(0,0,0,.25);transition:transform .25s var(--ease)}
.switch input:checked + .track{background:var(--accent)}
.switch input:checked + .track::after{transform:translateX(16px)}
.switch input:focus-visible + .track{box-shadow:0 0 0 4px var(--accent-soft)}
.seg{display:flex;align-items:center;gap:6px;margin-top:10px}
.seg .muted{margin-right:auto}
.seg button{appearance:none;border:1px solid var(--line);background:#fff;border-radius:6px;padding:3px 12px;cursor:pointer;transition:all .2s}
.seg button.is-active{background:var(--ink);border-color:var(--ink);color:#fff}
.row{display:flex;gap:10px;margin-top:12px}
.btn{appearance:none;border:1px solid var(--ink);background:var(--ink);color:#fff;border-radius:8px;padding:8px 16px;font-weight:500;cursor:pointer;transition:transform .15s var(--ease),background .2s}
.btn:hover{background:#1B3045} .btn:active{transform:scale(.97)}
.btn-ghost{background:#fff;color:var(--ink);border-color:var(--line)} .btn-ghost:hover{background:#F4F6F9}
.note{margin:12px 0 0;font-size:13px;color:var(--ok);min-height:20px}

/* analysis */
.line-input{width:100%;padding:12px 14px;border:1px solid var(--line);border-radius:8px;font:16px inherit;font-family:inherit;transition:border-color .2s,box-shadow .2s}
.tag-row{display:flex;flex-wrap:wrap;gap:10px;margin-top:18px;min-height:64px}
.tok{display:flex;flex-direction:column;align-items:center;gap:6px;animation:pop .35s var(--ease) both;animation-delay:calc(var(--i,0) * 45ms)}
.tok span:first-child{font-size:16px;font-weight:500}
.legend{display:flex;flex-wrap:wrap;gap:8px;margin-top:18px;padding-top:16px;border-top:1px solid var(--line)}

/* evaluation */
.metrics{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:16px;margin-bottom:20px}
.metric{background:var(--surface);border:1px solid var(--line);border-radius:var(--radius);padding:18px 20px;animation:rise .4s var(--ease) both;animation-delay:calc(var(--i,0) * 60ms)}
.metric .label{font-size:13px;color:var(--muted)}
.metric .value{font-size:30px;font-weight:600;letter-spacing:-.02em;margin-top:4px;font-variant-numeric:tabular-nums}
.table-wrap{overflow-x:auto}
table{width:100%;border-collapse:collapse;font-size:14px}
th{font-weight:500;color:var(--muted);text-align:right;padding:8px 12px;border-bottom:1px solid var(--line);white-space:nowrap}
td{padding:11px 12px;text-align:right;border-bottom:1px solid #EEF1F5;font-variant-numeric:tabular-nums}
th:first-child,td:first-child{text-align:left}
tbody tr{transition:background .2s} tbody tr:hover{background:#F7F9FB}
td.best{color:var(--accent);font-weight:600}
.cellbar{display:inline-block;width:80px;vertical-align:middle;margin-left:10px}

@keyframes rise{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:none}}
@keyframes pop{from{opacity:0;transform:scale(.9)}to{opacity:1;transform:none}}
@keyframes pulse{0%{box-shadow:0 0 0 0 rgba(14,124,102,.45)}70%,100%{box-shadow:0 0 0 7px rgba(14,124,102,0)}}

@media (max-width:920px){
  .layout,.grid-2{grid-template-columns:1fr}
  .brand-sub,.status{display:none}
  .topbar{padding:0 16px} main{padding:16px}
}
@media (prefers-reduced-motion:reduce){*{animation:none!important;transition:none!important}}

Writing web/style.css


In [26]:
%%writefile web/app.js
(() => {
"use strict";

const $ = (s, r = document) => r.querySelector(s);
const $$ = (s, r = document) => [...r.querySelectorAll(s)];
const reduce = matchMedia("(prefers-reduced-motion: reduce)").matches;

const store = {
  get(k) { try { return localStorage.getItem(k); } catch { return null; } },
  set(k, v) { try { localStorage.setItem(k, v); } catch { /* storage blocked */ } },
};
let uid = store.get("sk_uid");
if (!uid) { uid = crypto.randomUUID ? crypto.randomUUID() : String(Math.random()).slice(2); store.set("sk_uid", uid); }

async function call(path, body, signal) {
  const res = await fetch(path, {
    method: body ? "POST" : "GET",
    headers: { "Content-Type": "application/json" },
    body: body ? JSON.stringify(body) : undefined,
    signal,
  });
  if (!res.ok) throw new Error(path + " " + res.status);
  return res.json();
}
const debounce = (fn, ms) => { let t; return (...a) => { clearTimeout(t); t = setTimeout(() => fn(...a), ms); }; };
const pct = (x, d = 1) => (x == null ? "-" : (x * 100).toFixed(d) + "%");
const num = (x, d = 2) => (x == null ? "-" : x.toFixed(d));

const TAG_NAMES = { NOUN: "Noun", VERB: "Verb", ADJ: "Adjective", ADV: "Adverb", DET: "Determiner",
  ADP: "Preposition", PRON: "Pronoun", CONJ: "Conjunction", PRT: "Particle", NUM: "Number", X: "Other", END: "Sentence end" };

function tagEl(tag) {
  const s = document.createElement("span");
  s.className = "tag"; s.textContent = tag; s.dataset.tag = tag;
  return s;
}

function setStatus(ok, text) {
  const el = $("#status");
  el.classList.toggle("is-ok", ok); el.classList.toggle("is-bad", !ok);
  $("#statusText").textContent = text || (ok ? "Model ready" : "Offline");
}

/* ---------- tabs ---------- */
const tabs = $$(".tab"), ink = $("#tabInk");
function moveInk(el) { ink.style.width = el.offsetWidth + "px"; ink.style.transform = `translateX(${el.offsetLeft}px)`; }
function show(view) {
  const tab = tabs.find(t => t.dataset.view === view) || tabs[0];
  tabs.forEach(t => t.classList.toggle("is-active", t === tab));
  $$(".view").forEach(v => v.classList.toggle("is-active", v.id === "view-" + tab.dataset.view));
  moveInk(tab);
  if (tab.dataset.view === "analysis") runAnalysis();
  if (tab.dataset.view === "evaluation") loadMetrics();
  history.replaceState(null, "", "#" + tab.dataset.view);
}
tabs.forEach(t => t.addEventListener("click", () => show(t.dataset.view)));
addEventListener("resize", () => moveInk($(".tab.is-active")));

/* ---------- rank list helper (animated bars) ---------- */
function renderRank(ul, items) {
  ul.innerHTML = "";
  const max = Math.max(0.0001, ...items.map(i => i.value));
  items.forEach((it, n) => {
    const li = document.createElement("li");
    li.style.setProperty("--i", n);
    const label = document.createElement("span");
    label.className = "rank-label";
    label.append(document.createTextNode(it.label));
    if (it.tag) label.append(tagEl(it.tag));
    const bar = document.createElement("span"); bar.className = "bar";
    const fill = document.createElement("i"); bar.append(fill);
    const val = document.createElement("span"); val.className = "rank-val"; val.textContent = pct(it.value);
    li.append(label, bar, val); ul.append(li);
    requestAnimationFrame(() => requestAnimationFrame(() => fill.style.setProperty("--w", (it.value / max).toFixed(3))));
  });
}

/* ---------- writer ---------- */
const state = { k: 3, hmm: true, personal: true, seq: 0, current: [] };
const editor = $("#editor"), slotsEl = $("#slots"), phrasesEl = $("#phrases");
let slotEls = [], ctrl = null;

function buildSlots() {
  slotsEl.innerHTML = ""; slotEls = [];
  for (let i = 0; i < state.k; i++) {
    const b = document.createElement("button");
    b.type = "button"; b.className = "slot is-empty";
    b.innerHTML = '<span class="slot-top"><span class="slot-word"></span><span class="tag" hidden></span></span>' +
      '<span class="bar"><i></i></span>' +
      `<span class="slot-meta"><span class="slot-prob"></span><span>${i === 0 ? "Tab" : "Alt+" + (i + 1)}</span></span>`;
    b.addEventListener("mousedown", e => e.preventDefault());   // keep the caret in the editor
    b.addEventListener("click", () => accept(i));
    slotsEl.append(b); slotEls.push(b);
  }
}

function renderSlots(list) {
  state.current = list;
  slotEls.forEach((el, i) => {
    const s = list[i], w = $(".slot-word", el), t = $(".tag", el);
    el.classList.toggle("is-empty", !s);
    el.classList.toggle("is-personal", !!(s && s.personal));
    if (!s) { w.textContent = ""; t.hidden = true; $(".slot-prob", el).textContent = ""; $(".bar i", el).style.setProperty("--w", 0); return; }
    const label = s.eos ? "end of sentence" : s.word;
    if (w.textContent !== label) {
      w.textContent = label;
      if (!reduce) w.animate([{ opacity: 0, transform: "translateY(6px)" }, { opacity: 1, transform: "none" }], { duration: 200, easing: "ease-out" });
    }
    t.textContent = s.pos; t.dataset.tag = s.pos; t.hidden = false;
    $(".slot-prob", el).textContent = pct(s.prob);
    $(".bar i", el).style.setProperty("--w", Math.min(1, s.prob / Math.max(list[0].prob, 0.25)).toFixed(3));
  });
}

function renderPhrases(list) {
  phrasesEl.innerHTML = "";
  list.forEach((c, n) => {
    const li = document.createElement("li");
    const b = document.createElement("button");
    b.type = "button"; b.style.setProperty("--i", n);
    b.innerHTML = `<span></span><span class="ended">${c.ended ? "ends sentence" : ""}</span>`;
    b.firstChild.textContent = c.text;
    b.addEventListener("mousedown", e => e.preventDefault());
    b.addEventListener("click", () => insertAtCaret(c.text, c.ended));
    li.append(b); phrasesEl.append(li);
  });
}

function renderState(s) {
  $("#stMode").textContent = s.mode === "completion" ? `Completing "${s.prefix}"` : "Next word";
  $("#stOrder").textContent = s.order;
  $("#stLatency").textContent = s.latency_ms.toFixed(1) + " ms";
  const ctx = $("#stContext"); ctx.innerHTML = "";
  (s.context.length ? s.context : ["start of sentence"]).forEach((w, i) => {
    const c = document.createElement("span");
    c.className = "chip" + (s.oov.includes(w) ? " is-oov" : ""); c.textContent = w;
    c.title = s.oov.includes(w) ? "Not in the training vocabulary" : ""; c.style.animationDelay = i * 30 + "ms";
    ctx.append(c);
  });
  renderRank($("#nextTags"), s.next_tags.map(t => ({ label: TAG_NAMES[t.tag] || t.tag, value: t.p, tag: null })));
}

async function refresh() {
  const my = ++state.seq;
  if (ctrl) ctrl.abort();
  ctrl = new AbortController();
  const text = editor.value.slice(0, editor.selectionStart);
  $("#wordCount").textContent = (editor.value.trim() ? editor.value.trim().split(/\s+/).length : 0) + " words";
  try {
    const [s, c] = await Promise.all([
      call("/api/suggest", { text, k: state.k, user_id: uid, use_hmm: state.hmm, personalize: state.personal }, ctrl.signal),
      call("/api/complete", { text, length: 4, beam: 3, use_hmm: state.hmm }, ctrl.signal),
    ]);
    if (my !== state.seq) return;
    renderSlots(s.suggestions); renderPhrases(c.completions); renderState(s); setStatus(true);
  } catch (e) { if (e.name !== "AbortError") setStatus(false); }
}

function insertAtCaret(insert, endsSentence = false) {
  const pos = editor.selectionStart, v = editor.value;
  const before = v.slice(0, pos), after = v.slice(pos);
  const m = before.match(/[A-Za-z']+$/);
  let head = m ? before.slice(0, before.length - m[0].length) : before;
  if (/(^|[.!?]\s*)$/.test(head)) insert = insert.charAt(0).toUpperCase() + insert.slice(1);
  const added = head + insert + (endsSentence ? ". " : " ");
  editor.value = added + after.replace(/^ /, "");
  editor.setSelectionRange(added.length, added.length);
  editor.focus(); refresh();
}

function accept(i) {
  const s = state.current[i];
  if (!s) return;
  if (s.eos) {
    const pos = editor.selectionStart, before = editor.value.slice(0, pos).replace(/\s+$/, "");
    const added = before + ". ";
    editor.value = added + editor.value.slice(pos).replace(/^\s+/, "");
    editor.setSelectionRange(added.length, added.length); editor.focus(); refresh();
  } else insertAtCaret(s.word);
}

editor.addEventListener("input", debounce(refresh, 70));
editor.addEventListener("click", debounce(refresh, 30));
editor.addEventListener("keyup", e => { if (["ArrowLeft", "ArrowRight", "Home", "End"].includes(e.key)) refresh(); });
editor.addEventListener("keydown", e => {
  if (e.key === "Tab" && !e.shiftKey && state.current[0]) { e.preventDefault(); accept(0); }
  else if (e.altKey && /^Digit[1-5]$/.test(e.code)) { e.preventDefault(); accept(+e.code.slice(5) - 1); }
});

$("#optHmm").addEventListener("change", e => { state.hmm = e.target.checked; refresh(); });
$("#optPersonal").addEventListener("change", e => { state.personal = e.target.checked; refresh(); });
$$("#seg button").forEach(b => b.addEventListener("click", () => {
  $$("#seg button").forEach(x => x.classList.toggle("is-active", x === b));
  state.k = +b.dataset.k; buildSlots(); refresh();
}));

$("#btnLearn").addEventListener("click", async () => {
  const text = editor.value.trim();
  if (!text) { $("#profileNote").textContent = "Type some text first."; return; }
  try {
    const r = await call("/api/learn", { text, user_id: uid });
    $("#profileNote").textContent = `Learned ${r.tokens} words, ${r.n_new_words} new` +
      (r.new_words.length ? ` (${r.new_words.slice(0, 5).join(", ")})` : "") + `. Profile: ${r.profile_tokens} words.`;
    refresh();
  } catch { $("#profileNote").textContent = "Could not reach the server."; }
});
$("#btnForget").addEventListener("click", async () => {
  await call("/api/forget", { user_id: uid }).catch(() => {});
  $("#profileNote").textContent = "Profile reset."; refresh();
});

/* ---------- analysis ---------- */
async function runAnalysis() {
  const text = $("#analysisInput").value;
  try {
    const [plain, withHmm] = await Promise.all([
      call("/api/suggest", { text, k: 6, use_hmm: false, personalize: false }),
      call("/api/suggest", { text, k: 6, use_hmm: true, personalize: false }),
    ]);
    const row = $("#tagRow"); row.innerHTML = "";
    withHmm.tags.forEach((t, i) => {
      const d = document.createElement("div"); d.className = "tok"; d.style.setProperty("--i", i);
      const w = document.createElement("span"); w.textContent = t.word;
      d.append(w, tagEl(t.tag)); row.append(d);
    });
    if (!withHmm.tags.length) row.innerHTML = '<span class="muted">Type a sentence to see its part-of-speech tags.</span>';
    const toItems = r => r.suggestions.map(s => ({ label: s.eos ? "end of sentence" : s.word, value: s.prob, tag: s.pos }));
    renderRank($("#rankBase"), toItems(plain));
    renderRank($("#rankHmm"), toItems(withHmm));
  } catch { setStatus(false); }
}
$("#analysisInput").addEventListener("input", debounce(runAnalysis, 120));

/* ---------- evaluation ---------- */
let metrics = null;
function countUp(el, to, fmt, ms = 900) {
  if (reduce) { el.textContent = fmt(to); return; }
  const t0 = performance.now();
  const step = t => { const k = Math.min(1, (t - t0) / ms); el.textContent = fmt(to * (1 - Math.pow(1 - k, 3))); if (k < 1) requestAnimationFrame(step); };
  requestAnimationFrame(step);
}

async function loadMetrics() {
  if (metrics) return;
  try { metrics = await call("/api/metrics"); }
  catch { $("#metricCards").innerHTML = '<p class="muted">Run the evaluation step to create results/metrics.json.</p>'; return; }
  const rows = metrics.models, last = rows[rows.length - 1], d = metrics.dataset;

  const cards = [
    ["Perplexity", last.perplexity, v => v.toFixed(1)],
    ["Top-3 accuracy", last.top3 * 100, v => v.toFixed(1) + "%"],
    ["Keystroke savings", last.kss * 100, v => v.toFixed(1) + "%"],
    ["POS tagging accuracy", metrics.hmm.accuracy * 100, v => v.toFixed(1) + "%"],
    ["Latency per request", metrics.latency_ms, v => v.toFixed(1) + " ms"],
  ];
  const wrap = $("#metricCards"); wrap.innerHTML = "";
  cards.forEach(([label, value, fmt], i) => {
    const m = document.createElement("div"); m.className = "metric"; m.style.setProperty("--i", i);
    m.innerHTML = `<div class="label">${label}</div><div class="value">0</div>`;
    wrap.append(m); countUp($(".value", m), value, fmt);
  });

  $("#datasetLine").textContent = `${d.name} | ${d.train_tokens.toLocaleString()} training tokens | vocabulary ${d.vocab.toLocaleString()} | ${pct(d.oov_rate)} unknown words in test`;

  const cols = [["perplexity", "Perplexity", v => num(v), "min"], ["top1", "Top-1", v => pct(v), "max"],
    ["top3", "Top-3", v => pct(v), "max"], ["top5", "Top-5", v => pct(v), "max"],
    ["mrr", "MRR", v => num(v, 3), "max"], ["kss", "Keystroke savings", v => pct(v), "max"]];
  const best = {};
  cols.forEach(([k, , , dir]) => {
    const vals = rows.map(r => r[k]).filter(v => v != null);
    best[k] = dir === "min" ? Math.min(...vals) : Math.max(...vals);
  });
  $("#modelTable").innerHTML = "<thead><tr><th>Model</th>" + cols.map(c => `<th>${c[1]}</th>`).join("") + "</tr></thead><tbody>" +
    rows.map(r => `<tr><td>${r.name}</td>` + cols.map(([k, , f]) =>
      `<td class="${r[k] != null && r[k] === best[k] ? "best" : ""}">${r[k] == null ? "-" : f(r[k])}</td>`).join("") + "</tr>").join("") + "</tbody>";

  const sw = metrics.beta_sweep, maxT = Math.max(...sw.map(s => s.top3));
  $("#sweepTable").innerHTML = "<thead><tr><th>HMM weight</th><th>Perplexity</th><th>Top-3</th></tr></thead><tbody>" +
    sw.map(s => `<tr><td>${s.beta}${s.beta === metrics.best_beta ? " (selected)" : ""}</td><td class="${s.beta === metrics.best_beta ? "best" : ""}">${num(s.perplexity)}</td>` +
      `<td>${pct(s.top3)}<span class="bar cellbar"><i style="--w:${(s.top3 / maxT).toFixed(3)}"></i></span></td></tr>`).join("") + "</tbody>";

  const h = metrics.hmm;
  $("#posStats").innerHTML = [["Accuracy (Viterbi)", pct(h.accuracy)], ["Most-frequent-tag baseline", pct(h.baseline)],
    ["Known words", pct(h.accuracy_known)], ["Unknown words", pct(h.accuracy_unknown)],
    ["Unknown word share", pct(h.unknown_share)], ["Held-out tokens", h.n_tokens.toLocaleString()]]
    .map(([a, b]) => `<div><dt>${a}</dt><dd>${b}</dd></div>`).join("");
}

/* ---------- init ---------- */
(async function init() {
  buildSlots();
  try {
    const info = await call("/api/info");
    setStatus(true, "Model ready");
    const legend = $("#legend");
    info.tags.concat("END").forEach(t => { const s = tagEl(t); s.textContent = TAG_NAMES[t] || t; legend.append(s); });
  } catch { setStatus(false); }
  show((location.hash || "#writer").slice(1));
  refresh();
})();
})();

Writing web/app.js


## deployment files and run

In [27]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
numpy
joblib
pydantic

Overwriting requirements.txt


In [28]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 7860
CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT:-7860}"]

Writing Dockerfile


In [29]:
%%writefile .gitignore
.venv/
__pycache__/
.ipynb_checkpoints/
data/
*.pyc

Writing .gitignore


In [ ]:
uvicorn app.main:app --reload --port 8000
# open http://localhost:8000